# Hyperparameter Tuning of Machine Learning Models

Hyperparameter tuning is the process of finding the best hyperparameters for a machine learning model. 

## What are Hyperparameters?
- Hyperparameters are the parameters that are **not learned** by the model.
- They are set **before training** the model.
- The process of hyperparameter tuning is also known as **hyperparameter optimization**.

## Why is Hyperparameter Tuning Important?
- The performance of the model is highly dependent on the hyperparameters.
- The right choice of hyperparameters can significantly improve the model's performance.
- Hyperparameter tuning helps to:
  - Find the best hyperparameters for the model, resulting in optimal performance.
  - Improve the performance of the model.
  - Avoid **overfitting** and **underfitting**.
  - Make the model more robust.

## Techniques for Hyperparameter Tuning
Several techniques are available for hyperparameter tuning. Some of the most popular techniques in Scikit-learn are:
- **Grid Search**
- **Random Search**
- **Successive Halving**
- **Halving Grid Search**
- **Halving Random Search**

In [2]:
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import HalvingGridSearchCV, HalvingRandomSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Hyperparameter Tuning with scikit-learn on the Tips Dataset
# This notebook demonstrates how to perform hyperparameter tuning using scikit-learn's GridSearchCV on the Tips dataset.


# Load and Explore the Data
tips = sns.load_dataset('tips')
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [3]:
%%time
# Preprocess the Data
# Convert categorical variables using one-hot encoding
tips_encoded = pd.get_dummies(tips, drop_first=True)

# Define features and target variable
X = tips_encoded.drop('tip', axis=1)
y = tips_encoded['tip']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Hyperparameter Tuning with GridSearchCV
# Define the model
rf = RandomForestRegressor(random_state=42)

# Define the parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    # n_jobs=-1,
    scoring='neg_mean_squared_error'
)

# Fit GridSearchCV
grid_search.fit(X_train, y_train)

# Best Parameters and Evaluation
print(f"Best Parameters: {grid_search.best_params_}")

# Predict on test set
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)

# Calculate Mean Squared Error
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error on Test Set: {mse:.2f}")


Best Parameters: {'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 100}
Mean Squared Error on Test Set: 0.97
CPU times: total: 47.1 s
Wall time: 49.9 s


###  Explanation

1. **Preprocess the Data**
    - Converts categorical variables in the 'tips' dataset into numeric columns using one-hot encoding.
    - Drops the first category to avoid multicollinearity.

2. **Define features and target variable**
    - `X`: All columns except 'tip' (features).
    - `y`: The 'tip' column (target).

3. **Split the data**
    - Splits the data into training and testing sets (80% train, 20% test).

4. **Hyperparameter Tuning with GridSearchCV**
    - Sets up a `RandomForestRegressor`.
    - Defines a grid of hyperparameters to search over.
    - Uses `GridSearchCV` to find the best combination of hyperparameters using 5-fold cross-validation.

5. **Fit GridSearchCV**
    - Trains models for all combinations of hyperparameters on the training data.

6. **Best Parameters and Evaluation**
    - Prints the best hyperparameters found.
    - Uses the best model to predict on the test set.
    - Calculates and prints the mean squared error (MSE) on the test set.

In [4]:
%%time
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import RandomizedSearchCV, HalvingGridSearchCV, HalvingRandomSearchCV
# Preprocess the Data
# Convert categorical variables using one-hot encoding
tips_encoded = pd.get_dummies(tips, drop_first=True)

# Define features and target variable
X = tips_encoded.drop('tip', axis=1)
y = tips_encoded['tip']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Hyperparameter Tuning with GridSearchCV
# Define the model
rf = RandomForestRegressor(random_state=42)

# Define the parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

# Initialize RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=10,
    cv=5,
    # n_jobs=-1,
    scoring='neg_mean_squared_error'
)

# Fit RandomizedSearchCV
random_search.fit(X_train, y_train)

# Best Parameters and Evaluation
print(f"Best Parameters: {random_search.best_params_}")

# Predict on test set
best_rf = random_search.best_estimator_

y_pred = best_rf.predict(X_test)

# Calculate Mean Squared Error
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error on Test Set: {mse:.2f}")



Best Parameters: {'n_estimators': 100, 'min_samples_split': 10, 'max_depth': 30}
Mean Squared Error on Test Set: 0.97
CPU times: total: 15.2 s
Wall time: 15.6 s


### Explanation

1. **Imports**  
    - Imports necessary modules for halving and randomized search cross-validation from scikit-learn.

2. **Data Preprocessing**  
    - Converts categorical variables in the `tips` dataset into numeric columns using one-hot encoding.
    - Defines features (`X`) by dropping the 'tip' column and sets the target (`y`) as the 'tip' column.

3. **Train-Test Split**  
    - Splits the data into training and testing sets (80% train, 20% test).

4. **Model and Hyperparameter Grid**  
    - Initializes a `RandomForestRegressor` with a fixed random state for reproducibility.
    - Defines a parameter grid for hyperparameter tuning.

5. **RandomizedSearchCV Setup**  
    - Sets up `RandomizedSearchCV` to search over the parameter grid using 10 random combinations and 5-fold cross-validation.
    - Uses negative mean squared error as the scoring metric.

6. **Model Fitting and Evaluation**  
    - Fits the randomized search on the training data.
    - Prints the best hyperparameters found.
    - Uses the best model to predict on the test set.
    - Calculates and prints the mean squared error (MSE) on the test set.

In [5]:
%%time
# Initialize HalvingGridSearchCV
halving_grid_search = HalvingGridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    factor=2,
    # resource='n_estimators',
    scoring='neg_mean_squared_error'
)

# Fit HalvingGridSearchCV
halving_grid_search.fit(X_train, y_train)

# Best Parameters and Evaluation
print(f"Best Parameters: {halving_grid_search.best_params_}")

# Predict on test set
best_rf = halving_grid_search.best_estimator_
y_pred = best_rf.predict(X_test)

# Calculate Mean Squared Error
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error on Test Set: {mse:.2f}")

Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Mean Squared Error on Test Set: 0.94
CPU times: total: 1min 13s
Wall time: 1min 15s


### Explanation

1. **Imports**  
    - Imports modules required for halving and randomized search cross-validation from scikit-learn.

2. **Data Preprocessing**  
    - Converts categorical variables in the `tips` dataset into numeric columns using one-hot encoding.
    - Defines the features (`X`) by dropping the 'tip' column and sets the target (`y`) as the 'tip' column.

3. **Train-Test Split**  
    - Splits the data into training and testing sets (80% train, 20% test).

4. **Model and Hyperparameter Grid**  
    - Initializes a `RandomForestRegressor` with a fixed random state for reproducibility.
    - Defines a parameter grid for hyperparameter tuning.

5. **RandomizedSearchCV Setup**  
    - Sets up `RandomizedSearchCV` to search over the parameter grid using 10 random combinations and 5-fold cross-validation.
    - Uses negative mean squared error as the scoring metric.

6. **Model Fitting and Evaluation**  
    - Fits the randomized search on the training data.
    - Prints the best hyperparameters found.
    - Uses the best model to predict on the test set.
    - Calculates and prints the mean squared error (MSE) on the test set.

# About the Author

<div style="background-color: #f8f9fa; border-left: 5px solid #28a745; padding: 20px; margin-bottom: 20px; border-radius: 5px;">
  <h2 style="color: #28a745; margin-top: 0; font-family: 'Poppins', sans-serif;">Muhammad Atif Latif</h2>
  <p style="font-size: 16px; color: #495057;">Data Scientist & Machine Learning Engineer</p>
  
  <p style="font-size: 15px; color: #6c757d; margin-top: 15px;">
    Passionate about building AI solutions that solve real-world problems. Specialized in machine learning, 
    deep learning, and data analytics with experience implementing production-ready models.
  </p>
</div>

## Connect With Me

<div style="display: flex; flex-wrap: wrap; gap: 10px; margin-top: 15px;">
  <a href="https://github.com/m-Atif-Latif" target="_blank">
    <img src="https://img.shields.io/badge/GitHub-Follow-212121?style=for-the-badge&logo=github" alt="GitHub">
  </a>
  <a href="https://www.kaggle.com/matiflatif" target="_blank">
    <img src="https://img.shields.io/badge/Kaggle-Profile-20BEFF?style=for-the-badge&logo=kaggle" alt="Kaggle">
  </a>
  <a href="https://www.linkedin.com/in/muhammad-atif-latif-13a171318" target="_blank">
    <img src="https://img.shields.io/badge/LinkedIn-Connect-0077B5?style=for-the-badge&logo=linkedin" alt="LinkedIn">
  </a>
  <a href="https://x.com/mianatif5867" target="_blank">
    <img src="https://img.shields.io/badge/Twitter-Follow-1DA1F2?style=for-the-badge&logo=twitter" alt="Twitter">
  </a>
  <a href="https://www.instagram.com/its_atif_ai/" target="_blank">
    <img src="https://img.shields.io/badge/Instagram-Follow-E4405F?style=for-the-badge&logo=instagram" alt="Instagram">
  </a>
  <a href="mailto:muhammadatiflatif67@gmail.com">
    <img src="https://img.shields.io/badge/Email-Contact-D14836?style=for-the-badge&logo=gmail" alt="Email">
  </a>
</div>

---